# Expense Claim Analysis

এই নোটবুকটি দেখায় কীভাবে প্লাগইন ব্যবহার করে এজেন্ট তৈরি করা যায় যা লোকাল রসিদ ইমেজ থেকে ট্রাভেল খরচ প্রক্রিয়াজাত করে, একটি খরচ দাবি ইমেইল তৈরি করে এবং পাই চার্ট ব্যবহার করে খরচ ডেটা ভিজ্যুয়ালাইজ করে। এজেন্টগুলি কাজের প্রসঙ্গ অনুযায়ী ডাইনামিকভাবে ফাংশন নির্বাচন করে।

ধাপসমূহ:
1. OCR Agent লোকাল রসিদ ইমেজ প্রক্রিয়াজাত করে এবং ট্রাভেল খরচ সংক্রান্ত তথ্য আহরণ করে।
2. Email Agent একটি খরচ দাবি ইমেইল তৈরি করে।

### একটি ট্রাভেল খরচ পরিস্থিতির উদাহরণ:
ধরুন আপনি একজন কর্মী এবং একটি ব্যবসায়িক মিটিংয়ে অন্য শহরে যাচ্ছেন। আপনার কোম্পানির নীতি সব যুক্তিসঙ্গত ট্রাভেল-সম্পর্কিত খরচ ফেরত দেওয়া। এখানে সম্ভাব্য ট্রাভেল খরচের একটি ভাঙ্গন:
- পরিবহন:
আপনার গৃহ শহর থেকে গন্তব্য শহরে রাউন্ড ট্রিপের জন্য বিমানভাড়া।
এয়ারপোর্ট থেকে এবং এয়ারপোর্টে যাবার জন্য ট্যাক্সি বা রাইড-হেইলিং সেবা।
গন্তব্য শহরে স্থানীয় পরিবহন (যেমন পাবলিক ট্রানজিট, রেন্টাল গাড়ি, বা ট্যাক্সি)।

- থাকার জায়গা:
মিটিং এর স্থান থেকে কাছাকাছি একটি মধ্যম মানের ব্যবসায়িক হোটেলে তিন রাত থাকার খরচ।

- আহার:
প্রতিদিনের আহারের জন্য ব্রেকফাস্ট, লাঞ্চ এবং ডিনার, যা কোম্পানির পার ডায়েম নীতির উপর ভিত্তি করে দেয়া হয়।

- বৈচিত্র্যময় খরচ:
এয়ারপোর্টে পার্কিং ফি।
হোটেলে ইন্টারনেট এক্সেস চার্জ।
টিপস বা ছোট সার্ভিস চার্জ।

- ডকুমেন্টেশন:
আপনি সব রসিদ (ফ্লাইট, ট্যাক্সি, হোটেল, আহার ইত্যাদি) এবং একটি পূর্ণাঙ্গ খরচ রিপোর্ট জমা দেন ফেরতের জন্য।


## প্রয়োজনীয় লাইব্রেরি ইমপোর্ট করুন

নোটবুকের জন্য প্রয়োজনীয় লাইব্রেরি এবং মডিউলগুলি ইমপোর্ট করুন।


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## খরচ মডেল নির্ধারণ করুন

একটি পৃথক খরচের জন্য একটি পিড্যান্টিক মডেল তৈরি করুন এবং একটি ExpenseFormatter ক্লাস তৈরি করুন যা ব্যবহারকারীর প্রশ্নকে কাঠামোবদ্ধ খরচের তথ্যয়ে রূপান্তর করবে।

প্রত্যেকটি খরচ নিম্নলিখিত ফরম্যাটে উপস্থাপিত হবে:  
`{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## সরঞ্জাম সংজ্ঞায়িতকরণ - ইমেল তৈরি

খরচের দাবির জন্য একটি ইমেল তৈরি করার জন্য একটি টুল ফাংশন তৈরি করুন।  
- এই টুলটি Microsoft Agent Framework থেকে `@tool` ডেকোরেটর ব্যবহার করে।  
- এটি খরচের মোট পরিমাণ হিসাব করে এবং বিবরণগুলি একটি ইমেল বডিতে ফরম্যাট করে।


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# টুল টুলটি রসিদ চিত্র থেকে ভ্রমণ ব্যয় আহরণ করার জন্য

রসিদ চিত্র থেকে ভ্রমণ ব্যয় আহরণ করার জন্য একটি টুল ফাংশন তৈরি করুন।
- এই টুলটি Microsoft Agent Framework থেকে `@tool` ডেকোরেটর ব্যবহার করে।
- এটি রসিদ চিত্রটি পড়ে, এটি base64 হিসাবে এনকোড করে, এবং এজেন্টের বিশ্লেষণের জন্য ডেটা URI প্রদান করে।


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## প্রক্রিয়াকরণ ব্যয়

এজেন্টগুলি সংজ্ঞায়িত করুন এবং তাদের `WorkflowBuilder` ব্যবহার করে একটি ক্রমাগত ওয়ার্কফ্লোতে সংযুক্ত করুন।
- OCR এজেন্ট রসিদ চিত্র থেকে গঠনমূলক ব্যয় ডেটা বের করে `load_receipt_image` টুল ব্যবহার করে।
- Email এজেন্ট বের করা ডেটা গ্রহণ করে এবং একটি পেশাদার ব্যয় দাবির ইমেইল তৈরি করে `generate_expense_email` টুল ব্যবহার করে।
- `WorkflowBuilder` এর `add_edge` দিয়ে একটি ক্রমাগত পাইপলাইন তৈরি করে: OCR এজেন্ট → Email এজেন্ট।


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Main function

ক্রমিক ওয়ার্কফ্লো তৈরি করুন এবং চালান রসিদ চিত্র প্রক্রিয়া করার জন্য এবং খরচ দাবি ইমেল তৈরি করার জন্য।


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনলাইনে অনূদিত হয়েছে। আমরা সঠিকতার জন্য চেষ্টা করি, তবে স্বয়ংক্রিয় অনুবাদে ভুল বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার নিজস্ব ভাষায় মান্য উৎস হিসেবে গণ্য করা উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ প্রয়োজন। এই অনুবাদের ব্যবহারের ফলে যে কোনো ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়ী নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
